<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Conditional probability

**[C03-M03-L02 · Conditional probability](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/)** · C03, Essential Mathematics and Statistics for AI · Module 3, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** divide by the right group: P(urgent | payment) and P(payment | urgent) from the Tuesday table, then count the payment alert on 10,000 tickets and see why only 180 of its 670 flags are real.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | How to find a probability by counting, and joint events ("and") in the Tuesday table, from [Events and uncertainty: Joint events: "and" and "or"](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/#joint-events-and-and-or). Independence, from [Events and uncertainty: Independence: does one event change the other?](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/#independence-does-one-event-change-the-other). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M03-L02-conditional-probability/conditional-probability-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Probability inside a group

These are the 100 Tuesday tickets from the last lesson, as a **contingency table**: a count for each combination of two properties.

Run the cell. You should see the table with the totals 15, 85, 8, 92 and 100.

In [ ]:
#            urgent, not urgent
payment_row = [4, 11]
other_row = [4, 81]

table(["", "Urgent", "Not urgent", "Total"], [
    ["Payment ticket", *payment_row, sum(payment_row)],
    ["Other team", *other_row, sum(other_row)],
    ["Total", payment_row[0] + other_row[0], payment_row[1] + other_row[1], sum(payment_row) + sum(other_row)],
])

A **conditional probability** looks only at the cases in one group. P(urgent | payment), read "urgent, given payment", looks only at the payment row. P(payment | urgent) looks only at the urgent column.

> **Predict.** Both have the same top number, 4. Are the two probabilities equal? Then run the cell.

In [ ]:
both = 4
print("P(urgent | payment) = 4 / 15 =", round(both / 15, 2))
print("P(payment | urgent) = 4 / 8  =", round(both / 8, 2))

> **Check.** You should see `0.27` and `0.5`. The condition decides what you divide by: 15 payment tickets, or 8 urgent tickets.

## 2. The payment alert on 10,000 tickets

2 in 100 tickets are payment problems (the **base rate**). Tomás's keyword alert flags 90 of 100 real payment problems (the **recall**) and 5 of 100 other tickets (the **false-positive rate**). A wrong flag is a **false alarm**.

Priya counts 10,000 tickets, step by step. Whole numbers of tickets are easier to read than small decimals.

> **Predict.** Of all flagged tickets, what share is really a payment problem? Write a number, then run the cell.

In [ ]:
N = 10_000
base_rate, recall, false_alarm_rate = 0.02, 0.90, 0.05

problems = round(N * base_rate)
others = N - problems
real_flags = round(problems * recall)
missed = problems - real_flags
false_alarms = round(others * false_alarm_rate)
correctly_left = others - false_alarms
flags = real_flags + false_alarms

table(["Step", "Count"], [
    ["Payment problems (2%)", problems],
    ["Other tickets", others],
    ["Real flags (90% of problems)", real_flags],
    ["Missed", missed],
    ["False alarms (5% of others)", false_alarms],
    ["Correctly left", correctly_left],
    ["All flags", flags],
])
print()
print("share of flags that are real:", real_flags, "/", flags, "=", round(real_flags / flags, 3))

> **Check.** You should see 200, 9800, 180, 20, 490, 9310 and 670, and `180 / 670 = 0.269`. The lesson says about 27 in 100. The alert catches 9 in 10 problems, but only about 1 flag in 4 is real.

The lesson draws these counts as a frequency tree. The two flagged boxes of the tree are the real flags and the false alarms. Run the cell. You should see that the false-alarm bar is more than twice as long as the real-flag bar.

In [ ]:
bars(["Real flags", "False alarms"], [real_flags, false_alarms])

## 3. The same counts as a screening matrix

Each share divides by a different total. **Recall** divides by the payment-problem row. The share of flags that are real, called **precision**, divides by the flagged column. **Accuracy** divides by all 10,000 tickets.

> **Predict.** Which is larger: the accuracy or the recall? Then run the cell.

In [ ]:
table(["Share", "Calculation", "Value"], [
    ["Recall", "180 / 200", real_flags / problems],
    ["Precision (flags that are real)", "180 / 670", real_flags / flags],
    ["False-positive rate", "490 / 9,800", false_alarms / others],
    ["Accuracy", "(180 + 9,310) / 10,000", (real_flags + correctly_left) / N],
], digits=3)

> **Check.** You should see `0.900`, `0.269`, `0.050` and `0.949`. Accuracy is 94.9%, and still 73 of every 100 flags are wrong.

An alert that **never** flags anything is correct for all 9,800 other tickets. Run the cell. You should see `accuracy of "never flag": 0.98`, and it catches `0` payment problems.

In [ ]:
print('accuracy of "never flag":', others / N)
print('payment problems caught: ', 0)

## 4. Change the base rate

On the lesson page, the explorer has one slider: the base rate. Here you do the same with a list. The function counts 10,000 tickets for any base rate, recall and false-alarm rate. It rounds each count to a whole ticket, as the explorer does.

> **Predict.** During an outage, payment problems are 20% of tickets. Will most flags then be real? Then run the cell.

In [ ]:
def alert_counts(base_rate, recall=0.90, false_alarm_rate=0.05, n=10_000):
    """Count the real flags, false alarms, all flags and the share of flags that are real."""
    problems = round(n * base_rate)
    real = round(problems * recall)
    false = round((n - problems) * false_alarm_rate)
    share = real / (real + false) if real + false > 0 else None
    return real, false, real + false, share

rows = []
for rate in [0.0, 0.005, 0.02, 0.2]:
    real, false, all_flags, share = alert_counts(rate)
    rows.append([f"{rate:.1%}", real, false, all_flags, round(share, 3)])
table(["Base rate", "Real flags", "False alarms", "All flags", "Share real"], rows, digits=3)

> **Check.** You should see 0 of 500 flags real (0.000) at 0%, 45 of 543 (0.083) at 0.5%, 180 of 670 (0.269) at 2%, and 1800 of 2200 (0.818) at 20%. The alert is the same in every row. Only the base rate changes the meaning of a flag.

> **Your turn: change one thing.** In the cell above, add `0.5` to the list of base rates and run it again. At 50%, you should see 4500 real flags, 250 false alarms and a share of 0.947.

## 5. Bayes' rule: the same count as a formula

**Bayes' rule** writes the same steps as one line: P(problem | flag) = P(flag | problem) × P(problem) ÷ P(flag).

Run the cell. You should see a top line of `0.018`, P(flag) `0.067`, and the same `0.269` as the count.

In [ ]:
top = recall * base_rate                          # problem and flagged
p_flag = top + false_alarm_rate * (1 - base_rate)  # real flags + false alarms
table(["Part", "Value"], [
    ["P(problem), the base rate", base_rate],
    ["P(flag | problem), the recall", recall],
    ["top line: problem and flagged", top],
    ["P(flag): all flags", p_flag],
    ["P(problem | flag)", top / p_flag],
], digits=3)

The formula and the tree give the same answer, because they are the same count.

**Worked example: Priya answers Tomás.** 10,000 tickets are about two months of Larkfield's work: about 8.3 weeks of 1,200 tickets. Run the cell. You should see about `59` wrong tickets a week, which Priya rounds to 60.

In [ ]:
weeks = N / 1200
print("weeks in 10,000 tickets:", round(weeks, 1))
print("false alarms a week:    ", round(false_alarms / weeks))

## 6. Guided practice: a better keyword list

Mei removes phrases such as "free delivery charge". The new alert has the **same recall (90%)**, and the **false-alarm rate falls to 1 in 100**.

> **Your turn.** Count 10,000 tickets again with the new false-alarm rate. Replace each `...` with a calculation, then run the check cell. Do not use `alert_counts`: write each step.

In [ ]:
problems_g = 0.02 * 10_000
others_g = 10_000 - problems_g
real_g = 0.90 * problems_g
false_g = 0.01 * others_g
flags_g = real_g + false_g
share_g = real_g / flags_g

Run the check cell.

In [ ]:
expect("payment problems", problems_g, 200)
expect("other tickets", others_g, 9800)
expect("real flags", real_g, 180)
expect("false alarms", false_g, 98)
expect("all flags", flags_g, 278)
expect("the share of real flags", share_g, 0.647, tol=0.001)

About 2 of every 3 flags are now real, not 1 in 4. The recall did not change. With a rare problem, the false alarms decide what a flag means. Write one sentence for Grace in your `c03-notes.md`.

## 7. Independent variation: customers change their words

Change one condition: the alert now catches only **60 of 100** payment problems. The false-alarm rate is **5 in 100**, and 2% of tickets are payment problems.

> **Your turn.** Count 10,000 tickets. Calculate the real flags, the missed payment problems, the false alarms, all flags and the share of flags that are real. Run the check cell.

In [ ]:
real_v = 0.60 * 200
missed_v = 200 - real_v
false_v = 0.05 * 9800
flags_v = real_v + false_v
share_v = real_v / flags_v

Run the check cell.

In [ ]:
expect("real flags", real_v, 120)
expect("missed payment problems", missed_v, 80)
expect("false alarms", false_v, 490)
expect("all flags", flags_v, 610)
expect("the share of real flags", share_v, 0.197, tol=0.001)

The share of real flags falls from 27% to 20%, and the alert misses 80 payment problems instead of 20. The false alarms do not change, because they come from the other tickets.

## 8. Failure case: reading the alert's number the wrong way round

A plan says "the alert is 90% accurate, so 90% of flagged tickets are payment problems". The 90% is P(flag | payment problem). The plan needed the reverse.

Run the cell. You should see `0.9` and `0.269`: two different numbers with the same top count, 180.

In [ ]:
print("P(flag | payment problem) = 180 / 200 =", real_flags / problems)
print("P(payment problem | flag) = 180 / 670 =", round(real_flags / flags, 3))

> **Warning.** Before you use a number from a test, write it with the bar: P( … | … ). Ask which group it divides by. Then count 10,000 cases with the real base rate.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you divided by the right group: P(urgent | payment) = 4/15 = 0.27, but P(payment | urgent) = 4/8 = 0.5. On 10,000 tickets, the payment alert gives 180 real flags and 490 false alarms, so only 180 of 670 flags (0.269) are real, although the recall is 90% and the accuracy 94.9%. A lower false-alarm rate (1%) raises the share to 0.647. Bayes' rule is the same count as a formula.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Distributions in practice](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.